# Notebook 02 — QA Pair Generation (DeepSeek)

**Steps:**
1. Load chunks + parent_chunks từ NB01
2. Khởi tạo DeepSeek client
3. Generate QA pairs tự nhiên, đa dạng (prompt cải tiến)
4. Filter cặp chất lượng thấp
5. Dedup bằng cosine similarity
6. Train/test split 80/20 stratified theo source file
7. Lưu `qa_train.jsonl` + `qa_test.jsonl`

> **Prerequisite:** Run `01_setup_data_prep.ipynb` trước (cần `all_chunks.jsonl` + `parent_chunks.jsonl`).

In [ ]:
from pathlib import Path
import os

# Tự động tìm đường dẫn
if os.path.exists('/content/'):
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = Path('/content/drive/MyDrive/NLP_Final/Source/')
else:
    BASE = "../"

print(f'Base directory: {BASE}')
print(os.listdir(BASE))

Mounted at /content/drive
Base directory: /content/drive/MyDrive/NLP_Final/Source
['.gitignore', '.env', 'requirements.txt', 'scripts', '.git', 'data', 'notebooks', 'models', 'results']


## 1. Load Chunks

In [ ]:
import json, os

CHUNKS_PATH  = f'{BASE}/data/chunks/all_chunks.jsonl'
PARENTS_PATH = f'{BASE}/data/chunks/parent_chunks.jsonl'

# Load child chunks (dùng để tracking chunk_id)
all_chunks = []
with open(CHUNKS_PATH, 'r', encoding='utf-8') as f:
    for line in f:
        if line.strip():
            all_chunks.append(json.loads(line))

# Load parent chunks (dùng để sinh QA — nguyên vẹn ngữ nghĩa)
parent_map = {}   # parent_chunk_id → parent_text
with open(PARENTS_PATH, 'r', encoding='utf-8') as f:
    for line in f:
        if line.strip():
            p = json.loads(line)
            source_clean = p['source_file'].replace('.txt', '')
            parent_map[p['parent_chunk_id']] = {
                'source_file': source_clean,
                'text': p['text']
            }

print(f'Child chunks : {len(all_chunks)} từ {len(set(c["source_file"] for c in all_chunks))} files')
print(f'Parent chunks: {len(parent_map)}')

Child chunks : 850 từ 20 files
Parent chunks: 232


## 2. Khởi tạo DeepSeek Client

In [ ]:
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv(os.path.join(BASE, '.env'))
DEEPSEEK_API_KEY = os.environ.get('DEEPSEEK_API_KEY', 'YOUR_DEEPSEEK_KEY')

client   = OpenAI(api_key=DEEPSEEK_API_KEY, base_url='https://api.deepseek.com')
MODEL_ID = 'deepseek-chat'   # deepseek-chat = DeepSeek-V3 (tốt nhất, chi phí thấp)

# Kiểm tra kết nối
try:
    test = client.chat.completions.create(
        model=MODEL_ID,
        messages=[{'role': 'user', 'content': 'Chào'}],
        max_tokens=5,
    )
    print(f' DeepSeek connected. Model: {MODEL_ID}')
except Exception as e:
    print(f' Lỗi kết nối: {e}')

 DeepSeek connected. Model: deepseek-chat


## 3. QA Generation Function

In [ ]:
import time, random

SYSTEM_PROMPT = """Bạn là TDTU Slay - chuyên viên tư vấn sinh viên giàu kinh nghiệm của Trường Đại học Tôn Đức Thắng (TDTU).

Nhiệm vụ: Từ đoạn văn bản quy chế được cung cấp, tạo ra các cặp hỏi-đáp THẬT TỰ NHIÊN, gần gũi, đa dạng bằng tiếng Việt
dùng để huấn luyện chatbot tư vấn sinh viên.

--- QUAN TRỌNG: CÁCH NÓI TỰ NHIÊN NHƯ NGƯỜI THẬT ---
• CÂU HỎI: viết y hệt như một sinh viên đang thắc mắc thật sự.
  - Được phép dùng: "ạ", "dạ", "cho em hỏi", "em hơi lo", "bọn em", "vậy ạ", "cụ thể là…?"
  - KHÔNG được bắt đầu bằng: "Theo văn bản…", "Mục đích của hướng dẫn là…", "Văn bản nêu rõ…"
  - Ví dụ hay: "Nếu em lỡ… thì có sao không ạ?" / "Mấy cái này áp dụng cho ai vậy?"
• CÂU TRẢ LỜI: như một anh/chị đi trước đang giải thích cặn kẽ, thân thiện.
   - Mở đầu linh hoạt: "Chào em!", "Dạ vâng!", "Có chứ em…", "Ừ đúng rồi…", "Rõ ràng thế này nhé…"
  - KHÔNG dùng: "Theo văn bản…", "Văn bản nêu rõ…", "Mục đích của hướng dẫn là…"
  - Nội dung chính: đầy đủ số liệu, điều kiện, không suy diễn.
  - Kết thúc: có thể thêm một câu dặn dò thân thiện, linh hoạt, KHÔNG bắt buộc.
    Ví dụ (chỉ là gợi ý, không phải danh sách giới hạn):
    • "Có gì chưa rõ hỏi lại tụi anh nha."
    • "Em lưu ý giúp nhé."
    • "Chúc em học tốt!"
    • "Nếu còn thắc mắc cứ hỏi tiếp nha."
    • "Nhớ đọc kỹ để tránh rủi ro không đáng có nhé."
    → Tùy ngữ cảnh, có thể không cần câu kết nếu câu trả lời đã đủ tự nhiên.

--- YÊU CẦU VỀ CÂU HỎI ---
• Đa dạng kiểu câu hỏi — xoay vòng các dạng sau, không lặp lại cùng 1 dạng:
  [Điều kiện]  "Cần đáp ứng những tiêu chí gì để ...?"
  [Thủ tục]    "Làm thế nào / Các bước để...?"
  [Thời hạn]   "Khi nào / Trong bao lâu...?"
  [Hậu quả]    "Nếu vi phạm ... thì bị xử lý như thế nào?"
  [Tình huống] "Em đang ở tình huống ..., em có thể ...?"
  [Xác nhận]   "... có được phép / được tính / được xét không?"
  [So sánh]    "Sự khác nhau giữa ... và ... là gì ạ?"
  [Số lần]     "Em vi phạm lần đầu và lần thứ 2 có khác gì về mức xử lý không ạ?"
• Ngôn ngữ: tự nhiên như sinh viên thật sự hỏi, không hỏi theo kiểu văn bản pháp lý
sinh viên thắc mắc - chatbox trả lời chứ không được sinh viên đã biết chatbox trả lời lại
• Câu hỏi phải trả lời được hoàn toàn từ đoạn văn bản đã cho
• Khai thác Bảng biểu: Nếu văn bản có dạng bảng, phải tạo ít nhất một câu hỏi riêng biệt cho từng hàng hoặc từng đối tượng cụ thể trong bảng.

--- YÊU CẦU VỀ CÂU TRẢ LỜI ---
• Đầy đủ tất cả thông tin liên quan trong đoạn văn (không bỏ sót điều kiện, mức xử lý, thời hạn)
• Giọng văn thân thiện, dễ hiểu, mở đầu linh hoạt có lời chào, kết thúc có thể có thêm lời dặn dò phù hợp — như nhân viên tư vấn đang giải thích cho sinh viên
• Giữ nguyên số liệu cụ thể (%, điểm, số tiền, thời gian) từ văn bản gốc
• Độ dài: 40–300 từ
• Không suy diễn: Nếu văn bản không nêu, tuyệt đối không tự ý thêm thông tin từ các nguồn ngoài.
• CẤU TRÚC CHO CÂU DÀI (>150 từ): Sử dụng 1 trong các cách sau:
  [A] Dạng bullet points: mỗi ý một dấu • hoặc 📌
  [B] Dạng đánh số: 1 2 3 cho quy trình có thứ tự
  [C] Dạng phân loại: Theo đối tượng (sinh viên/giảng viên) hoặc theo trường hợp
  [D] Dạng tiêu đề phụ: **Về điều kiện:** ... **Về hồ sơ:** ...

--- TRƯỜNG HỢP ĐẶC BIỆT ---
• Nếu đoạn văn bản chỉ có tiêu đề, mục lục hoặc không chứa nội dung quy định cụ thể: Trả về JSON trống {"qa_pairs": []}
• Nếu đoạn văn bản quá ngắn (<50 ký tự) và không có thông tin: Trả về JSON trống

--- HƯỚNG DẪN THỰC HIỆN ---
1. Đọc kỹ nội dung chunk được cung cấp
2. Xác định các đối tượng, sự kiện, con số, điều kiện trong chunk
3. Với mỗi thông tin quan trọng, tạo 1 câu hỏi tự nhiên
4. Viết câu trả lời đầy đủ, trích dẫn đúng số liệu
5. Kiểm tra lại tính chính xác và tính tự nhiên

--- ĐỊNH DẠNG TRẢ VỀ ---
Chỉ trả về JSON hợp lệ, không có text nào khác:
{"qa_pairs": [{"question": "...", "answer": "..."}]}"""

def generate_qa_for_chunk(chunk: dict, n_pairs: int = 3, max_retries: int = 3) -> list[dict]:
    pid = chunk.get('parent_chunk_id')
    parent_info = parent_map.get(pid)

    if parent_info:
        source_name = parent_info['source_file']   # Lấy source từ dict
        parent_text = parent_info['text']          # Lấy text từ dict
    else:
        # Fallback nếu không có parent
        source_name = chunk['source_file'].replace('.txt', '')
        parent_text = chunk['text']

    user_message = (
        f"Đoạn văn bản quy chế TDTU:\n"
        f"Tên văn bản: {source_name}\n"
        f"Nội dung:\n"
        f"---\n{parent_text}\n---\n\n"
        f"Hãy tạo {n_pairs} cặp hỏi-đáp đa dạng, tự nhiên từ đoạn văn bản trên.\n"
        f"Lưu ý: Mỗi câu hỏi phải tự đứng vững, tuyệt đối không dùng các từ (quy luật 'này/trên/dưới đây') để chỉ văn bản."
    )

    for attempt in range(max_retries):
        try:
            resp = client.chat.completions.create(
                model=MODEL_ID,
                messages=[
                    {'role': 'system', 'content': SYSTEM_PROMPT},
                    {'role': 'user',   'content': user_message},
                ],
                response_format={'type': 'json_object'},
                temperature=0.85,
                max_tokens=4096,   # tăng để đủ cho 12 cặp QA
            )
            data = json.loads(resp.choices[0].message.content)
            pairs = data.get('qa_pairs', [])
            result = []
            for p in pairs:
                if isinstance(p, dict) and 'question' in p and 'answer' in p:
                    result.append({
                        'question'       : str(p['question']).strip(),
                        'answer'         : str(p['answer']).strip(),
                        'chunk_id'       : chunk['chunk_id'],
                        'parent_chunk_id': chunk.get('parent_chunk_id', ''),
                        'source_file'    : chunk['source_file'],
                    })
            return result
        except Exception as e:
            wait = (2 ** attempt) + random.uniform(0, 1)
            print(f'  Attempt {attempt+1} failed ({e}), retry {wait:.1f}s...')
            time.sleep(wait)
    return []


print(' QA generation function ready.')
print('   Standalone question constraint + max_tokens=4096 for long chunks.')

 QA generation function ready.
   Standalone question constraint + max_tokens=4096 for long chunks.


## 4. Generate QA Pairs (với Resume Support)

In [ ]:
from tqdm import tqdm

RAW_QA_PATH = f'{BASE}/data/qa_raw/qa_raw.jsonl'

# Resume: load parent_ids đã generate
generated_parent_ids = set()
if os.path.exists(RAW_QA_PATH):
    with open(RAW_QA_PATH, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                rec = json.loads(line)
                if rec.get('parent_chunk_id'):
                    generated_parent_ids.add(rec['parent_chunk_id'])
    print(f'Resume: {len(generated_parent_ids)} parents đã xử lý.')
else:
    print('Starting fresh generation.')

# Build danh sách parents cần xử lý
seen_parents = {}
for chunk in all_chunks:
    pid = chunk.get('parent_chunk_id')
    if pid and pid not in seen_parents:
        seen_parents[pid] = chunk

parents_to_process = [
    chunk for pid, chunk in seen_parents.items()
    if pid not in generated_parent_ids
]
print(f'Parents cần xử lý: {len(parents_to_process)} / {len(seen_parents)}')


def n_pairs_for_parent(parent_text: str) -> int:
    """
    Số QA tỷ lệ với độ dài — đảm bảo bao quát hết nội dung của chunk dài.
    Chunk 6000 chars có thể chứa 7-9 chủ đề → cần nhiều câu hỏi hơn.
    """
    n = len(parent_text)
    if n > 5000: return 12
    if n > 3000: return 9
    if n > 1500: return 6
    if n > 600:  return 4
    return 3


# Generate
qa_counter = 0
errors     = 0

with open(RAW_QA_PATH, 'a', encoding='utf-8') as f_out:
    for chunk in tqdm(parents_to_process, desc='Generating QA from parents'):
        pid = chunk['parent_chunk_id']
        parent_info = parent_map.get(pid)
        if parent_info:
            parent_text = parent_info['text']
        else:
            parent_text = chunk['text']

        n = n_pairs_for_parent(parent_text)
        pairs = generate_qa_for_chunk(chunk, n_pairs=n)
        if pairs:
            for pair in pairs:
                pair['parent_chunk_id'] = pid
                f_out.write(json.dumps(pair, ensure_ascii=False) + '\n')
            qa_counter += len(pairs)
        else:
            errors += 1
        time.sleep(0.5)

print(f'\n Generated {qa_counter} QA pairs | Errors: {errors}')

Resume: 231 parents đã xử lý.
Parents cần xử lý: 1 / 232


Generating QA from parents: 100%|██████████| 1/1 [00:05<00:00,  5.33s/it]


 Generated 3 QA pairs | Errors: 0


## 5. Load All Raw QA Pairs

In [ ]:
raw_pairs = []
if not os.path.exists(RAW_QA_PATH):
    print('qa_raw.jsonl chưa tồn tại — chạy cell Generate trước.')
else:
    with open(RAW_QA_PATH, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                raw_pairs.append(json.loads(line))
    print(f'Total raw QA pairs: {len(raw_pairs)}')

Total raw QA pairs: 1533


## 6. Filter Low-Quality Pairs

In [ ]:
chunk_map = {c['chunk_id']: c for c in all_chunks}

# Phrases that signal meta-textual / referential language (forbidden by system prompt)
_FORBIDDEN_Q = [
    'theo văn bản', 'văn bản nêu', 'văn bản này', 'theo đoạn văn',
    'mục đích của hướng dẫn', 'cơ sở pháp lý', 'điều khoản thi hành',
    'theo đoạn trên', 'nội dung trên cho thấy',
]
_FORBIDDEN_A = [
    'theo văn bản', 'văn bản nêu rõ', 'văn bản quy định',
    'mục đích của hướng dẫn',
]
# Dangling references make a question non-standalone
_DANGLING_REFS = [
    'quy định này áp dụng', 'điều này áp dụng', 'điều kiện trên',
    'văn bản trên', 'nội dung trên', 'hướng dẫn này áp dụng',
    'đoạn văn bản', 'theo đó,',
]

def get_trigrams(text: str) -> set:
    words = text.lower().split()
    return set(zip(words, words[1:], words[2:])) if len(words) >= 3 else set()

def is_valid(pair: dict) -> tuple[bool, str]:
    q = pair.get('question', '').strip()
    a = pair.get('answer', '').strip()

    # 1. Minimum character length
    if len(q) < 15:
        return False, 'q_too_short'
    if len(a) < 40:
        return False, 'a_too_short'

    # 2. Minimum word count
    if len(q.split()) < 5:
        return False, 'q_few_words'
    if len(a.split()) < 10:
        return False, 'a_few_words'

    # 3. Forbidden / meta-textual phrases
    q_low = q.lower()
    a_low = a.lower()
    if any(ph in q_low for ph in _FORBIDDEN_Q):
        return False, 'q_forbidden_phrase'
    if any(ph in a_low for ph in _FORBIDDEN_A):
        return False, 'a_forbidden_phrase'

    # 4. Non-standalone question (dangling reference to the source text)
    if any(ref in q_low for ref in _DANGLING_REFS):
        return False, 'q_dangling_ref'

    # 5. Trigram overlap: answer must share content with its source chunk
    chunk = chunk_map.get(pair.get('chunk_id'))
    if chunk:
        pid         = chunk.get('parent_chunk_id')
        parent_info = parent_map.get(pid)
        # parent_map stores dicts {'source_file': ..., 'text': ...}
        if isinstance(parent_info, dict):
            ref_text = parent_info['text']
        elif isinstance(parent_info, str):
            ref_text = parent_info
        else:
            ref_text = chunk['text']
        if len(get_trigrams(ref_text) & get_trigrams(a)) < 2:
            return False, 'low_trigram_overlap'

    return True, 'ok'

results  = [is_valid(p) for p in raw_pairs]
filtered = [p for p, (ok, _) in zip(raw_pairs, results) if ok]

from collections import Counter as _Ctr
reason_counts = _Ctr(reason for ok, reason in results if not ok)
rejected = len(raw_pairs) - len(filtered)
print(f'After filtering: {len(filtered)} / {len(raw_pairs)} kept  '
      f'({rejected} rejected = {rejected/len(raw_pairs)*100:.1f}%)')
if reason_counts:
    print('Rejection reasons:')
    for reason, count in reason_counts.most_common():
        print(f'  {reason:25s}: {count}')

After filtering: 1487 / 1533 kept  (46 rejected = 3.0%)
Rejection reasons:
  low_trigram_overlap      : 38
  a_forbidden_phrase       : 4
  q_forbidden_phrase       : 2
  q_dangling_ref           : 2


## 7. Deduplicate với Embedding Similarity

In [ ]:
%%capture
from sentence_transformers import SentenceTransformer
import numpy as np

dedup_model = SentenceTransformer('bkai-foundation-models/vietnamese-bi-encoder')
print('Embedding model loaded.')

In [ ]:
questions = [p['question'] for p in filtered]
q_embeddings = dedup_model.encode(questions, batch_size=64, normalize_embeddings=True, show_progress_bar=True)

SIMILARITY_THRESHOLD = 0.92
keep = []
kept_embeddings = []

for pair, emb in zip(filtered, q_embeddings):
    is_dup = any(float(np.dot(emb, ke)) > SIMILARITY_THRESHOLD for ke in kept_embeddings)
    if not is_dup:
        keep.append(pair)
        kept_embeddings.append(emb)

print(f'After dedup: {len(keep)} pairs (removed {len(filtered) - len(keep)} duplicates)')

Batches:   0%|          | 0/24 [00:00<?, ?it/s]

After dedup: 1458 pairs (removed 29 duplicates)


## 8. Train / Test Split (80/20 Stratified theo Source File)

Toàn bộ 20 file đều có đại diện trong train và test.  
FAISS index chứa chunk của tất cả 20 file → retriever hoạt động đúng với mọi câu hỏi test.

In [ ]:
from collections import defaultdict
import random
random.seed(42)

# Group theo source_file để stratify
by_file = defaultdict(list)
for p in keep:
    by_file[p['source_file']].append(p)

train_pairs, test_pairs = [], []

for source_file, pairs in by_file.items():
    random.shuffle(pairs)
    n_test = max(1, int(len(pairs) * 0.20))
    test_pairs.extend(pairs[:n_test])
    train_pairs.extend(pairs[n_test:])

# Assign ID + split label
# Gold key cho Recall@5 là parent_chunk_id (QA sinh từ parent, không phải child)
for i, p in enumerate(train_pairs):
    p['id']             = f'qa_{i+1:04d}'
    p['split']          = 'train'
    p['human_verified'] = False
    # Đảm bảo parent_chunk_id luôn có mặt (gold key cho evaluation)
    p.setdefault('parent_chunk_id', '')

for i, p in enumerate(test_pairs):
    p['id']             = f'qa_test_{i+1:04d}'
    p['split']          = 'test'
    p['human_verified'] = False
    p.setdefault('parent_chunk_id', '')

print(f'Train: {len(train_pairs)} | Test: {len(test_pairs)}')
print(f'Files covered in test: {len(set(p["source_file"] for p in test_pairs))}/20')
print('Train ≥300.' if len(train_pairs) >= 300 else f' Train chỉ {len(train_pairs)}, cần ≥300.')
print('Test  ≥50.'  if len(test_pairs)  >= 50  else f' Test  chỉ {len(test_pairs)}, cần ≥50.')

Train: 1174 | Test: 284
Files covered in test: 20/20
Train ≥300.
Test  ≥50.


## 9. Save QA Datasets

In [ ]:
TRAIN_PATH = f'{BASE}/data/qa_filtered/qa_train.jsonl'
TEST_PATH  = f'{BASE}/data/qa_filtered/qa_test.jsonl'

with open(TRAIN_PATH, 'w', encoding='utf-8') as f:
    for p in train_pairs:
        f.write(json.dumps(p, ensure_ascii=False) + '\n')

with open(TEST_PATH, 'w', encoding='utf-8') as f:
    for p in test_pairs:
        f.write(json.dumps(p, ensure_ascii=False) + '\n')

print(f'Saved {len(train_pairs)} train pairs → {TRAIN_PATH}')
print(f'Saved {len(test_pairs)} test pairs  → {TEST_PATH}')

Saved 1174 train pairs → /content/drive/MyDrive/NLP_Final/Source/data/qa_filtered/qa_train_2.jsonl
Saved 284 test pairs  → /content/drive/MyDrive/NLP_Final/Source/data/qa_filtered/qa_test_2.jsonl


## 10. Statistics & Sample Preview

In [ ]:
all_pairs = train_pairs + test_pairs
if not all_pairs:
    print('-- Không có pairs — kiểm tra các bước trước.')
else:
    print('=== QA Dataset Statistics ===')
    print(f'Train: {len(train_pairs)} | Test: {len(test_pairs)}')

    q_lens = [len(p['question'].split()) for p in all_pairs]
    a_lens = [len(p['answer'].split()) for p in all_pairs]
    print(f'\nQuestion (words): avg={sum(q_lens)/len(q_lens):.1f}  min={min(q_lens)}  max={max(q_lens)}')
    print(f'Answer   (words): avg={sum(a_lens)/len(a_lens):.1f}  min={min(a_lens)}  max={max(a_lens)}')

    print('\nSource file coverage trong test set:')
    test_by_file = defaultdict(int)
    for p in test_pairs:
        test_by_file[p['source_file']] += 1
    for fname, cnt in sorted(test_by_file.items(), key=lambda x: -x[1]):
        print(f'  {cnt:3d}  {fname}')

    print('\n--- Sample train pair ---')
    s = random.choice(train_pairs)
    print(f'Q: {s["question"]}')
    print(f'A: {s["answer"][:300]}')
    print(f'Parent: {s.get("parent_chunk_id", "N/A")}')

    print('\n--- Sample test pair ---')
    s = random.choice(test_pairs)
    print(f'Q: {s["question"]}')
    print(f'A: {s["answer"][:300]}')
    print(f'Parent: {s.get("parent_chunk_id", "N/A")}')

=== QA Dataset Statistics ===
Train: 1174 | Test: 284

Question (words): avg=20.7  min=9  max=45
Answer   (words): avg=71.1  min=10  max=248

Source file coverage trong test set:
   54  Hướng dẫn học bổng, khen thưởng, hỗ trợ người học và hỗ trợ khác.txt
   49  2021_Quy chế tổ chức và quản lý đào tạo.txt
   42  Quy chế Công tác sinh viên.txt
   22  QĐ số 22 vv ban hành quy định kiểm soát và xử lý hành vi đạo văn các sản phẩm học thuật.txt
   17  Quy chế đánh giá kết quả rèn luyện.txt
   13  Bộ Quy tắc ứng xử của người học.txt
   12  17.Thong tin phong ban.txt
   11  Quy dinh cong nhan ket qua hoc chi va chuyen doi tin chi.txt
    9  25.2020-Noi quy phong thi.txt
    9  QD 2793 - Nội quy thi trực tuyến 19.9.2023.txt
    8  Nội dung và thang điểm đánh giá rèn luyện trình độ đại học.txt
    7  (TV)-Hướng dẫn giao tiếp, ứng xử cho người học.txt
    7  15.2018 - Quy dinh ve hoat dong tap su nghe nghiep.txt
    7  Quy định cử nhân ưu tú_ap dung T9.2023.txt
    5  11.2457-QD cap chung nhan ky

In [ ]:

# Conversation type hints added to the user message to maximise diversity across runs
CONV_STYLE_HINTS = [
    "Sinh viên bắt đầu bằng [Thủ tục]: hỏi quy trình/các bước thực hiện, rồi đào sâu điều kiện, thời hạn và ngoại lệ.",
    "Sinh viên gặp [Tình huống] cụ thể lo lắng, TDTU Slay hướng dẫn từng bước; sinh viên xác nhận thêm ngoại lệ.",
    "Sinh viên hỏi [Xác nhận] quyền lợi, sau đó [So sánh] sự khác biệt giữa các trường hợp/đối tượng.",
    "Sinh viên hỏi [Hậu quả] vi phạm, mức xử lý lần 1 và lần 2 khác nhau thế nào, rồi hỏi cách khắc phục.",
    "Sinh viên hỏi [Điều kiện] xét duyệt, rồi [Thời hạn] và trường hợp đặc biệt/được miễn.",
]

CONV_SYSTEM_PROMPT = """Bạn là TDTU Slay - chuyên viên tư vấn sinh viên giàu kinh nghiệm của Trường Đại học Tôn Đức Thắng (TDTU).

Nhiệm vụ: Dựa trên đoạn văn bản quy chế TDTU được cung cấp, tạo một cuộc hội thoại THẬT TỰ NHIÊN, MẠCH LẠC giữa một sinh viên và TDTU Slay dùng để huấn luyện chatbot tư vấn.

--- QUAN TRỌNG: CÁCH NÓI TỰ NHIÊN NHƯ NGƯỜI THẬT ---
• LƯỢT SINH VIÊN: viết y hệt như một sinh viên đang thắc mắc thật sự.
  - Được phép dùng: "ạ", "dạ", "cho em hỏi", "em hơi lo", "vậy ạ", "Còn nếu…", "Vậy thì…", "Em chưa hiểu rõ…"
  - KHÔNG được bắt đầu bằng: "Theo văn bản…", "Mục đích của hướng dẫn là…", "Văn bản nêu rõ…"
  - Ví dụ hay: "Dạ vậy nếu em bị… thì có sao không ạ?" / "Còn trường hợp… thì sao ạ?"
  - SAI: câu quá ngắn như "Điểm F thì sao?" — phải có ngữ cảnh và cảm xúc tự nhiên
  - Mỗi lượt tiếp theo LIÊN KẾT với câu trả lời trước — không nhảy chủ đề đột ngột
• LƯỢT TDTU SLAY: như một anh/chị đi trước đang giải thích cặn kẽ, thân thiện.
  - Mở đầu linh hoạt: "Chào em!", "Dạ vâng!", "Có chứ em…", "Ừ đúng rồi…", "Rõ ràng thế này nhé…"
  - KHÔNG dùng: "Theo văn bản…", "Văn bản nêu rõ…", "Mục đích của hướng dẫn là…"
  - Nội dung chính: đầy đủ số liệu, điều kiện, không suy diễn ngoài văn bản
  - Kết thúc: có thể thêm câu dặn dò thân thiện phù hợp ngữ cảnh, KHÔNG bắt buộc.
    Ví dụ (chỉ là gợi ý): "Có gì chưa rõ hỏi lại tụi anh nha." / "Chúc em học tốt!" / "Nếu còn thắc mắc cứ hỏi tiếp nha."

--- CẤU TRÚC HỘI THOẠI ---
• Lượt 1 (user): câu hỏi MỞ tự nhiên về chủ đề chính trong văn bản
• Lượt tiếp (user): đào sâu — tình huống cụ thể, ngoại lệ, thủ tục chi tiết, hậu quả, so sánh
• Bắt đầu bằng user, kết thúc bằng assistant, xen kẽ đều đặn
• KHÔNG nhảy chủ đề đột ngột giữa các lượt

--- YÊU CẦU VỀ ĐA DẠNG CÂU HỎI ---
• Xoay vòng các dạng câu hỏi xuyên suốt hội thoại, không lặp cùng 1 dạng:
  [Điều kiện]  "Em cần đáp ứng những tiêu chí gì để...?"
  [Thủ tục]    "Quy trình / các bước thực hiện là...?"
  [Thời hạn]   "Khi nào / Trong bao lâu...?"
  [Hậu quả]    "Nếu vi phạm ... thì bị xử lý như thế nào?"
  [Tình huống] "Dạ em đang trong tình huống..., em có thể... không ạ?"
  [Xác nhận]   "... có được phép / được tính / được xét không ạ?"
  [So sánh]    "Sự khác nhau giữa ... và ... là gì ạ?"
• Câu hỏi phải trả lời được hoàn toàn từ đoạn văn bản đã cho
• Câu hỏi phải TỰ ĐỨNG VỮNG — tuyệt đối không dùng "này/trên/dưới đây" để chỉ văn bản

--- YÊU CẦU VỀ CÂU TRẢ LỜI (TDTU SLAY) ---
• Đầy đủ tất cả thông tin liên quan (không bỏ sót điều kiện, mức xử lý, thời hạn)
• Giữ nguyên số liệu cụ thể (%, điểm, số tiền, thời gian) từ văn bản gốc
• Không suy diễn: Nếu văn bản không nêu, tuyệt đối không tự ý thêm thông tin từ nguồn ngoài
• Độ dài: 40–300 từ mỗi lượt
• CẤU TRÚC CHO CÂU DÀI (>150 từ): Sử dụng 1 trong các cách sau:
  [A] Dạng bullet points: mỗi ý một dấu • hoặc 📌
  [B] Dạng đánh số: 1 2 3 cho quy trình có thứ tự
  [C] Dạng phân loại: Theo đối tượng (sinh viên/giảng viên) hoặc theo trường hợp
  [D] Dạng tiêu đề phụ: **Về điều kiện:** ... **Về hồ sơ:** ...

--- TRƯỜNG HỢP ĐẶC BIỆT ---
• Nếu văn bản chỉ có tiêu đề, mục lục hoặc không có nội dung quy định cụ thể: Trả về {"topic": "N/A", "turns": []}
• Nếu văn bản quá ngắn (<50 ký tự) và không có thông tin: Trả về {"topic": "N/A", "turns": []}

--- HƯỚNG DẪN THỰC HIỆN ---
1. Đọc kỹ văn bản, xác định chủ đề chính và các thông tin quan trọng
2. Lên kịch bản hội thoại tự nhiên (sinh viên hỏi từ tổng quát → cụ thể)
3. Viết từng lượt, đảm bảo mạch hội thoại liên kết và tự nhiên
4. Kiểm tra: số liệu chính xác, không suy diễn, giọng văn tự nhiên

--- ĐỊNH DẠNG TRẢ VỀ ---
Chỉ trả về JSON hợp lệ, không có text nào khác ngoài JSON:
{"topic": "Tên chủ đề ngắn gọn 5-10 từ", "turns": [{"role": "user", "content": "..."}, {"role": "assistant", "content": "..."}, ...]}"""


def generate_conversation(chunk_texts: list[str],
                          source_file: str = '',
                          n_turns: int | None = None,
                          max_retries: int = 3) -> dict | None:
    """Generate a multi-turn conversation for a set of semantically related chunks."""
    if n_turns is None:
        # Bias toward 4-6 turns; 8 turns for richer chunks
        n_turns = random.choices([4, 6, 8], weights=[3, 3, 1])[0]
    n_turns = max(4, min(8, n_turns if n_turns % 2 == 0 else n_turns + 1))

    style = random.choice(CONV_STYLE_HINTS)
    context = '\n\n---\n\n'.join(chunk_texts)

    src_label = f"Tên văn bản: {source_file}\n" if source_file else ""
    user_msg = (
        f"Đoạn văn bản quy chế TDTU:\n"
        f"{src_label}"
        f"Nội dung:\n---\n{context}\n---\n\n"
        f"Phong cách hội thoại gợi ý: {style}\n\n"
        f"Hãy tạo cuộc hội thoại ĐÚNG {n_turns} lượt "
        f"(user bắt đầu, assistant kết thúc, xen kẽ đều đặn) "
        f"xoay quanh nội dung văn bản trên.\n"
        f"Lưu ý: câu hỏi phải tự đứng vững, tuyệt đối không dùng 'này/trên/dưới đây' để chỉ văn bản."
    )

    for attempt in range(max_retries):
        try:
            resp = client.chat.completions.create(
                model=MODEL_ID,
                messages=[
                    {'role': 'system', 'content': CONV_SYSTEM_PROMPT},
                    {'role': 'user',   'content': user_msg},
                ],
                response_format={'type': 'json_object'},
                temperature=0.90,
                max_tokens=4096,
            )
            data  = json.loads(resp.choices[0].message.content)
            turns = data.get('turns', [])
            topic = str(data.get('topic', 'Quy định TDTU')).strip()

            # -- Validation
            if len(turns) < 4:
                raise ValueError(f'Only {len(turns)} turns — need ≥4')
            for t in turns:
                if t.get('role') not in ('user', 'assistant'):
                    raise ValueError(f'Unexpected role: {t.get("role")}')
                if len(str(t.get('content', '')).strip()) < 8:
                    raise ValueError('Turn content too short')
            if turns[0]['role'] != 'user':
                raise ValueError('First turn must be user')
            if turns[-1]['role'] != 'assistant':
                raise ValueError('Last turn must be assistant')

            # Trim to ≤8 turns (keep balanced user/assistant pairs)
            if len(turns) > 8:
                turns = turns[:8]
                if turns[-1]['role'] != 'assistant':
                    turns = turns[:-1]

            return {'topic': topic, 'turns': turns}

        except Exception as e:
            wait = (2 ** attempt) + random.uniform(0, 1)
            print(f'  Attempt {attempt+1} failed ({e}). Retry in {wait:.1f}s…')
            time.sleep(wait)
    return None

print('Conversation generation function ready.')
print(f'Style pool: {len(CONV_STYLE_HINTS)} diversity hints loaded.')

Conversation generation function ready.
Style pool: 5 diversity hints loaded.


In [ ]:

import numpy as np
from collections import defaultdict

# Rebuild seen_parents if this cell is run independently
try:
    _ = seen_parents
    print(f'Using seen_parents from Section 4: {len(seen_parents)} parents')
except NameError:
    seen_parents = {}
    for chunk in all_chunks:
        pid = chunk.get('parent_chunk_id')
        if pid and pid not in seen_parents:
            seen_parents[pid] = chunk
    print(f'Rebuilt seen_parents: {len(seen_parents)} parents')

# Lazy-load embedding model
try:
    _ = dedup_model
    print('Reusing dedup_model from Section 7.')
except NameError:
    from sentence_transformers import SentenceTransformer
    dedup_model = SentenceTransformer('bkai-foundation-models/vietnamese-bi-encoder')
    print('Loaded dedup_model fresh.')

CONV_PATH = f'{BASE}/data/qa_filtered/conversations_train.jsonl'
os.makedirs(os.path.dirname(CONV_PATH), exist_ok=True)

# Resume: load already-generated cluster keys
generated_keys: set[str] = set()
if os.path.exists(CONV_PATH):
    with open(CONV_PATH, 'r', encoding='utf-8') as _f:
        for _line in _f:
            if _line.strip():
                _rec = json.loads(_line)
                generated_keys.add(_rec.get('cluster_key', ''))
    print(f'Resume: {len(generated_keys)} clusters already done.')
else:
    print('Starting fresh.')

# Build eligible parent list (text > 100 chars)
all_parents: list[dict] = [
    {'parent_chunk_id': pid,
     'source_file'    : chunk['source_file'],
     'text'           : parent_map[pid]['text'] if pid in parent_map else chunk['text']}
    for pid, chunk in seen_parents.items()
    if len(parent_map[pid]['text'] if pid in parent_map else chunk['text']) > 100
]

by_file: dict[str, list[dict]] = defaultdict(list)
for p in all_parents:
    by_file[p['source_file']].append(p)

print(f'Eligible parents: {len(all_parents)} across {len(by_file)} files')

# Encode all parent texts for clustering
print('Encoding parent chunks…')
all_texts      = [p['text'] for p in all_parents]
all_embeddings = dedup_model.encode(
    all_texts, batch_size=64, normalize_embeddings=True, show_progress_bar=True)
emb_map: dict[str, np.ndarray] = {
    p['parent_chunk_id']: emb
    for p, emb in zip(all_parents, all_embeddings)
}

#  Per-file conversation cap — scale theo tổng độ dài text
# Phân phối thực tế (total_chars/file, 20 files):
#   Rất dài  > 40 000 chars: 83 419, 62 060, 43 864  (3 files)
#   Dài      > 10 000 chars: 30 217 → 10 249          (10 files)
#   Ngắn    ≤ 10 000 chars:  7 669 → 1 302            (7 files)
def get_max_conv_per_file(chunks: list[dict], base_limit: int = 5) -> int:
    """Tăng giới hạn cho file có tổng độ dài text lớn."""
    total_chars = sum(len(c['text']) for c in chunks)
    if total_chars > 40_000:
        return min(base_limit * 3, 15)   # Rất dài (>40 K chars): tối đa 15 conversations
    elif total_chars > 10_000:
        return min(base_limit * 2, 10)   # Dài (10–40 K chars): tối đa 10 conversations
    else:
        return base_limit                # Ngắn (≤10 K chars): giữ nguyên 5 conversations

#  Greedy cosine clustering within each source_file
CLUSTER_SIM  = 0.62   # chunks với cosine sim ≥ này → gộp vào cùng topic cluster
MAX_CHUNK_CL = 3      # max chunks/cluster (giới hạn độ dài context gửi lên API)

def cluster_chunks(chunks: list[dict]) -> list[list[dict]]:
    remaining = list(chunks)
    clusters  = []
    while remaining:
        seed     = remaining.pop(0)
        cluster  = [seed]
        seed_emb = emb_map[seed['parent_chunk_id']]
        leftover = []
        for c in remaining:
            if (len(cluster) < MAX_CHUNK_CL
                    and float(np.dot(seed_emb, emb_map[c['parent_chunk_id']])) >= CLUSTER_SIM):
                cluster.append(c)
            else:
                leftover.append(c)
        clusters.append(cluster)
        remaining = leftover
    return clusters

all_clusters: list[dict] = []
for src_file, chunks in by_file.items():
    clusters   = cluster_chunks(chunks)
    file_limit = get_max_conv_per_file(chunks)   # dynamic cap theo total_chars
    if len(clusters) > file_limit:
        random.seed(42)
        clusters = random.sample(clusters, file_limit)
    for cl in clusters:
        ckey = '_'.join(sorted(c['parent_chunk_id'] for c in cl))
        all_clusters.append({'source_file': src_file, 'chunks': cl, 'cluster_key': ckey})

to_process = [cl for cl in all_clusters if cl['cluster_key'] not in generated_keys]

#  Summary trước khi generate
from collections import Counter as _Counter
per_file_count = _Counter(cl['source_file'] for cl in all_clusters)
print(f'\nTotal clusters planned : {len(all_clusters)}')
print(f'Already generated      : {len(generated_keys)}')
print(f'Remaining to process   : {len(to_process)}')
print('\nPlanned conversations per file:')
for fname, cnt in sorted(per_file_count.items(), key=lambda x: -x[1]):
    print(f'  {cnt:3d}  {fname}')

#  Generate loop
conv_counter = 0
errors       = 0

with open(CONV_PATH, 'a', encoding='utf-8') as f_out:
    for cluster in tqdm(to_process, desc='Generating conversations'):
        chunk_texts = [c['text'] for c in cluster['chunks']]
        pids        = [c['parent_chunk_id'] for c in cluster['chunks']]

        result = generate_conversation(chunk_texts, source_file=cluster['source_file'])
        if result:
            record = {
                'conversation_id' : f'conv_{conv_counter + len(generated_keys) + 1:04d}',
                'source_file'     : cluster['source_file'],
                'parent_chunk_ids': pids,
                'cluster_key'     : cluster['cluster_key'],
                'topic'           : result['topic'],
                'n_turns'         : len(result['turns']),
                'turns'           : result['turns'],
            }
            f_out.write(json.dumps(record, ensure_ascii=False) + '\n')
            conv_counter += 1
        else:
            errors += 1
        time.sleep(0.5)

print(f'\nDone. Generated {conv_counter} new conversations | Errors: {errors}')
print(f'Total in file: {conv_counter + len(generated_keys)}')
print(f'Saved to: {CONV_PATH}')


Using seen_parents from Section 4: 232 parents
Reusing dedup_model from Section 7.
Starting fresh.
Eligible parents: 231 across 20 files
Encoding parent chunks…


Batches:   0%|          | 0/4 [00:00<?, ?it/s]


Total clusters planned : 135
Already generated      : 0
Remaining to process   : 135

Planned conversations per file:
   15  2021_Quy chế tổ chức và quản lý đào tạo.txt
   15  Hướng dẫn học bổng, khen thưởng, hỗ trợ người học và hỗ trợ khác.txt
   15  Quy chế Công tác sinh viên.txt
   10  17.Thong tin phong ban.txt
   10  Quy chế đánh giá kết quả rèn luyện.txt
   10  QĐ số 22 vv ban hành quy định kiểm soát và xử lý hành vi đạo văn các sản phẩm học thuật.txt
    7  QD 2793 - Nội quy thi trực tuyến 19.9.2023.txt
    6  (TV)-Hướng dẫn giao tiếp, ứng xử cho người học.txt
    6  Bộ Quy tắc ứng xử của người học.txt
    6  Quy dinh cong nhan ket qua hoc chi va chuyen doi tin chi.txt
    5  15.2018 - Quy dinh ve hoat dong tap su nghe nghiep.txt
    5  25.2020-Noi quy phong thi.txt
    5  MÔ TẢ 5 ĐẶC ĐIỂM NHẬN DIỆN SINH VIÊN TDTU.txt
    5  Nội dung và thang điểm đánh giá rèn luyện trình độ đại học.txt
    5  Quy định cử nhân ưu tú_ap dung T9.2023.txt
    4  11.2457-QD cap chung nhan ky su cu 

Generating conversations:  59%|█████▊    | 79/135 [13:39<09:54, 10.61s/it]

  Attempt 1 failed (Last turn must be assistant). Retry in 2.0s…


Generating conversations: 100%|██████████| 135/135 [23:43<00:00, 10.55s/it]


Done. Generated 135 new conversations | Errors: 0
Total in file: 135
Saved to: /content/drive/MyDrive/NLP_Final/Source/data/qa_filtered/conversations_train_2.jsonl


In [ ]:

# Load generated conversations
conversations_preview = []
if os.path.exists(CONV_PATH):
    with open(CONV_PATH, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                conversations_preview.append(json.loads(line))

if not conversations_preview:
    print('No conversations found. Run the generation cell first.')
else:
    from collections import Counter

    n_turns_list = [len(c['turns']) for c in conversations_preview]
    print('=== Conversation Dataset Statistics ===')
    print(f'Total conversations : {len(conversations_preview)}')
    print(f'Turns/conversation  : avg={sum(n_turns_list)/len(n_turns_list):.1f}  '
          f'min={min(n_turns_list)}  max={max(n_turns_list)}')
    print(f'Turn distribution   : {dict(sorted(Counter(n_turns_list).items()))}')

    total_turns = sum(n_turns_list)
    print(f'Total turns         : {total_turns} '
          f'(≈{total_turns//2} user-assistant pairs for training)')

    print('\nCoverage by source_file:')
    by_src = Counter(c['source_file'] for c in conversations_preview)
    for fname, cnt in sorted(by_src.items(), key=lambda x: -x[1]):
        print(f'  {cnt:3d}  {fname}')

    sample = random.choice(conversations_preview)
    print(f'\n--- Sample Conversation ---')
    print(f'Topic  : {sample["topic"]}')
    print(f'Source : {sample["source_file"]}')
    print(f'Turns  : {len(sample["turns"])}')
    for t in sample['turns']:
        role_label = 'Student ' if t['role'] == 'user' else 'TDTU Slay '
        content = t['content']
        preview = content[:250] + ('…' if len(content) > 250 else '')
        print(f'\n{role_label}: {preview}')


=== Conversation Dataset Statistics ===
Total conversations : 135
Turns/conversation  : avg=5.7  min=4  max=8
Turn distribution   : {4: 60, 6: 34, 7: 1, 8: 40}
Total turns         : 771 (≈385 user-assistant pairs for training)

Coverage by source_file:
   15  2021_Quy chế tổ chức và quản lý đào tạo.txt
   15  Hướng dẫn học bổng, khen thưởng, hỗ trợ người học và hỗ trợ khác.txt
   15  Quy chế Công tác sinh viên.txt
   10  17.Thong tin phong ban.txt
   10  Quy chế đánh giá kết quả rèn luyện.txt
   10  QĐ số 22 vv ban hành quy định kiểm soát và xử lý hành vi đạo văn các sản phẩm học thuật.txt
    7  QD 2793 - Nội quy thi trực tuyến 19.9.2023.txt
    6  (TV)-Hướng dẫn giao tiếp, ứng xử cho người học.txt
    6  Bộ Quy tắc ứng xử của người học.txt
    6  Quy dinh cong nhan ket qua hoc chi va chuyen doi tin chi.txt
    5  15.2018 - Quy dinh ve hoat dong tap su nghe nghiep.txt
    5  25.2020-Noi quy phong thi.txt
    5  MÔ TẢ 5 ĐẶC ĐIỂM NHẬN DIỆN SINH VIÊN TDTU.txt
    5  Nội dung và thang điể

---
**Next steps:**
- Run `03_rag_pipeline.ipynb` để build FAISS vector store từ toàn bộ chunks.
